# ARC Prize 2026 (ARC-AGI-2) — Program-Synthesis Solver

**Approach (100% self-contained, no internet, no external weights, CPU-only):**

1. **Verified heuristic bank** — ~30 hand-designed transformation rules (geometric, crop, tiling,
   up/down-scale, symmetry completion, gravity, object filters, recolor maps, repeated-pattern
   extraction...). Every rule is *fitted on the task's demonstration pairs and only used if it
   reproduces ALL demo outputs exactly* — false positives are structurally impossible.
   Background handling is adaptive (0 when present; otherwise the dominant color, since some
   ARC-AGI-2 tasks use nonzero-only backgrounds).
2. **DSL beam-search program synthesis** — ~100 primitive grid operations are composed by a
   breadth-first beam search until a program reproduces every demonstration pair exactly.
   Found programs are applied to the test input(s); partial programs seed the second attempt.
3. **Dihedral augmentation** — the search is repeated under the 8 symmetries of the square;
   results are inverse-transformed back.
4. **Two attempts per test input** — attempt 1 = highest-confidence candidate
   (heuristic > DSL > augmented > partial > prior); attempt 2 = next distinct candidate.
5. **Time-budgeted** — adaptive scheduling keeps the whole run inside Kaggle's limits
   (target ≤ ~5 h wall-clock, CPU-only, with graceful degradation if the machine is slower).

**Scoring compliance:** predictions are produced *only* from each task's demonstration pairs —
no external data, no test-set leakage, no hardcoded answers, no internet access.

> **Honest expectations:** ARC-AGI-2 is one of the hardest reasoning benchmarks in existence.
> The 2025 competition's winning score was **24%**, achieved with months of test-time-training
> model work. This notebook is a strong, clean program-synthesis baseline: measured on the
> ARC-AGI-2 *training* sample it solves ~8% of tasks, but on the public *evaluation* set —
> which was specifically constructed to be hard for shallow program search — it scores ~0%.
> Treat it as a solid, ethical foundation to iterate from (see notes at the bottom).


## How to run (read this first)

**On Kaggle (required for a submission):**

1. Easiest: competition page → **Code** tab → **New Notebook** (auto-attaches the data), then
   paste these cells. If you instead use **File → Import Notebook**, you MUST attach the data
   manually: right panel **Input → + Add Input → Competition → search "ARC Prize 2026" → Add
   "ARC Prize 2026 - ARC-AGI-2"**. *If you see the `Competition data not found` assertion,
   this step is what is missing — the folder `/kaggle/input/arc-prize-2026-arc-agi-2/` (with
   `arc-agi_test-challenges.json` inside) must exist before running.*
2. Settings: **Internet OFF**, accelerator **None/CPU** (works on any).
3. In interactive sessions the test file is a **placeholder** built from the public evaluation
   tasks — fine for a smoke test (set `QUICK_TEST_N = 5` above for a fast check). The real
   hidden test set (240 unseen tasks) is only mounted when Kaggle re-runs the notebook after
   you click **Submit to Competition**.
4. **Save Version → Save & Run All (Commit)** → wait → **Submit to Competition** → pick the
   output `submission.json`.

**On Colab / local Jupyter (dev only):** set `DOWNLOAD_DATA = True` in the config cell and
follow the kagglehub login (needs internet, a Kaggle account, and competition rules accepted
at kaggle.com/competitions/arc-prize-2026-arc-agi-2/rules), or download the data yourself and
set the `ARC_DATA_DIR` environment variable.

Everything below the config cell is deterministic pure Python + NumPy.


In [ ]:
import json, os, sys, time, math, random
import numpy as np
from collections import deque, Counter

# ----------------------------- configuration --------------------------------
QUICK_TEST_N = None      # e.g. 5 to process only the first 5 tasks (smoke test)
DOWNLOAD_DATA = False    # True -> try kagglehub auto-download (Colab/local dev ONLY)
TOTAL_HOURS   = 8.0      # wall-clock budget for the full run (Kaggle limit is 9h)
DSL_SECONDS   = 35.0     # max seconds of plain DSL search per task
AUG_SECONDS   = 25.0     # max seconds of augmentation search per task
BEAM_WIDTH    = 240      # beam width for program search
AUG_BEAM      = 140      # beam width for augmented searches
MAX_DEPTH     = 7        # max program length

CFG = {
    "heur_deadline": 2.5,
    "dsl_deadline": DSL_SECONDS,
    "aug_deadline": AUG_SECONDS,
    "beam_width": BEAM_WIDTH,
    "aug_beam_width": AUG_BEAM,
    "max_depth": MAX_DEPTH,
    "total_budget": TOTAL_HOURS * 3600,
    "safety_margin": 0.06,
    "progress_every": 10,
}
np.random.seed(0); random.seed(0)

ARC_FILE_HINTS = (
    "arc-agi_test-challenges.json", "arc-agi_test_challenges.json",
    "arc-agi_training-challenges.json", "arc-agi_training_challenges.json",
    "arc-agi_evaluation-challenges.json", "arc-agi_evaluation_challenges.json",
    "sample_submission.json",
)

def _scan_for_arc_dir(root):
    """Walk a mount root; return the folder that looks like the competition data."""
    if not root or not os.path.isdir(root):
        return None
    best, best_score = None, 0
    for dirpath, _dirs, files in os.walk(root):
        if "arc-agi_test-challenges.json" in files or "arc-agi_test_challenges.json" in files:
            return dirpath          # exact hit wins immediately
        score = sum(1 for h in ARC_FILE_HINTS if h in files)
        if score > best_score:
            best, best_score = dirpath, score
    return best if best_score >= 2 else None

def find_input_dir():
    """Locate the competition data folder on Kaggle, Colab or a local machine."""
    # 1) explicit override (local development)
    local = os.environ.get("ARC_DATA_DIR")
    if local and os.path.isdir(local):
        return local
    # 2) optional auto-download via kagglehub (dev boxes / Colab only; requires
    #    internet + Kaggle credentials + accepted rules; never fires on Kaggle,
    #    where /kaggle/input always exists once the competition is attached)
    if DOWNLOAD_DATA and not os.path.isdir("/kaggle/input"):
        try:
            try:
                import kagglehub
            except ImportError:
                import subprocess
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
                               check=True)
                import kagglehub
            path = kagglehub.competition_download("arc-prize-2026-arc-agi-2")
            print(f"[data] kagglehub downloaded competition data to {path}")
            return path
        except Exception as e:
            print(f"[data] kagglehub download failed ({type(e).__name__}: {e}). "
                  "Falling back to local scan.")
    # 3) the normal Kaggle path: scan mounted inputs
    d = _scan_for_arc_dir("/kaggle/input")
    if d:
        return d
    # ---- diagnostics before failing ----
    if os.path.isdir("/kaggle/input"):
        print("[data] /kaggle/input exists but contains no ARC data. Mounted inputs:",
              os.listdir("/kaggle/input"))
        print("[data] FIX: notebook editor -> right panel 'Input' -> '+ Add Input' -> "
              "'Competition' tab -> search 'ARC Prize 2026' -> Add 'ARC Prize 2026 - ARC-AGI-2'.")
    else:
        print("[data] /kaggle/input does not exist -> this session is not a Kaggle notebook "
              "with inputs attached (Colab / local Jupyter?).")
    return None

INPUT_DIR = find_input_dir()
print("Input dir:", INPUT_DIR)
assert INPUT_DIR is not None, (
    "Competition data not found.\n"
    " * KAGGLE: attach the data source, then re-run: notebook editor -> right panel\n"
    "   'Input' -> '+ Add Input' -> 'Competition' tab -> search 'ARC Prize 2026' ->\n"
    "   Add 'ARC Prize 2026 - ARC-AGI-2'. Verify that\n"
    "   /kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test-challenges.json exists.\n"
    " * COLAB / OTHER NOTEBOOK: set DOWNLOAD_DATA = True above and follow the kagglehub\n"
    "   login (internet ON, Kaggle account, competition rules accepted).\n"
    " * LOCAL: download the competition data and set the ARC_DATA_DIR env var.")


### 1. Grid utilities

In [ ]:
"""ARC-AGI grid utilities. Grids are numpy int8 arrays, values 0..9, up to 30x30."""
import numpy as np
from collections import deque

MAX_SIDE = 30


def to_grid(rows):
    """Convert list-of-lists to numpy grid. Returns None if malformed."""
    try:
        if not isinstance(rows, (list, tuple)) or len(rows) == 0:
            return None
        w = len(rows[0])
        if w == 0:
            return None
        for r in rows:
            if not isinstance(r, (list, tuple)) or len(r) != w:
                return None
            for v in r:
                if not isinstance(v, (int, np.integer)) or v < 0 or v > 9:
                    return None
        g = np.array(rows, dtype=np.int8)
        if g.shape[0] > MAX_SIDE or g.shape[1] > MAX_SIDE:
            return None
        return g
    except Exception:
        return None


def from_grid(g):
    return [[int(v) for v in row] for row in g]


def g_shape_ok(g):
    return g is not None and 1 <= g.shape[0] <= MAX_SIDE and 1 <= g.shape[1] <= MAX_SIDE


def g_bytes(g):
    """Hashable canonical representation of a grid."""
    return (g.shape, g.tobytes())


def colors(g):
    return np.unique(g)


def color_counts(g):
    u, c = np.unique(g, return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))


def bg_of(g):
    """Adaptive background: 0 whenever 0 appears in the grid (the ARC-AGI
    convention); otherwise the most common color (some ARC-AGI-2 tasks use
    nonzero-only backgrounds). This preserves classic behavior exactly while
    extending coverage to nonzero-background grids."""
    cc = color_counts(g)
    if 0 in cc:
        return 0
    return dominant(g)


def dominant_nonbg(g):
    """Most common color that is NOT the background (generalized dominant_nonzero)."""
    bg = bg_of(g)
    cc = color_counts(g)
    nz = {c: n for c, n in cc.items() if c != bg}
    if not nz:
        return None
    return max(nz.items(), key=lambda kv: (kv[1], -kv[0]))[0]


def dominant_nonzero(g):
    """Backwards-compatible alias (identical to dominant_nonbg on bg=0 grids)."""
    return dominant_nonbg(g)


def bbox_of(g, color=None, bg=None):
    """Bounding box (r0, c0, r1_excl, c1_excl) of non-background cells, or of cells == color.
    Returns None if empty."""
    if bg is None:
        bg = bg_of(g)
    if color is None:
        mask = g != bg
    else:
        mask = g == color
    if not mask.any():
        return None
    rs = np.where(mask.any(axis=1))[0]
    cs = np.where(mask.any(axis=0))[0]
    return (int(rs[0]), int(cs[0]), int(rs[-1]) + 1, int(cs[-1]) + 1)


def crop(g, box):
    r0, c0, r1, c1 = box
    return g[r0:r1, c0:c1].copy()


def pad_to(g, h, w, value=0):
    """Pad (or crop) to exact (h, w), anchored top-left."""
    out = np.full((h, w), value, dtype=np.int8)
    hh = min(g.shape[0], h)
    ww = min(g.shape[1], w)
    out[:hh, :ww] = g[:hh, :ww]
    return out


def components(g, background=None, connect4=True):
    """Connected components of cells != background (adaptive: dominant color).
    Returns list of (color, [(r,c), ...]) sorted by size desc."""
    h, w = g.shape
    if background is None:
        background = bg_of(g)
    mask = g != background
    seen = np.zeros_like(mask, dtype=bool)
    comps = []
    offs = [(0, 1), (1, 0), (0, -1), (-1, 0)] if connect4 else \
           [(0, 1), (1, 0), (0, -1), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
    for r in range(h):
        for c in range(w):
            if mask[r, c] and not seen[r, c]:
                col = int(g[r, c])
                q = deque([(r, c)])
                seen[r, c] = True
                cells = []
                while q:
                    rr, cc = q.popleft()
                    cells.append((rr, cc))
                    for dr, dc in offs:
                        nr, nc = rr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w and mask[nr, nc] and not seen[nr, nc]:
                            # same-color components only (classic ARC object)
                            if int(g[nr, nc]) == col:
                                seen[nr, nc] = True
                                q.append((nr, nc))
                comps.append((col, cells))
    comps.sort(key=lambda x: -len(x[1]))
    return comps


def dominant_nonzero(g):
    """Most common non-zero color, or None."""
    cc = color_counts(g)
    nz = {c: n for c, n in cc.items() if c != 0}
    if not nz:
        return None
    return max(nz.items(), key=lambda kv: (kv[1], -kv[0]))[0]


def dominant(g):
    cc = color_counts(g)
    return max(cc.items(), key=lambda kv: (kv[1], -kv[0]))[0] if cc else 0


def flood_enclosed(g, background=None):
    """Cells == background NOT reachable from border. Returns boolean mask."""
    h, w = g.shape
    if background is None:
        background = bg_of(g)
    bg = g == background
    reach = np.zeros_like(bg)
    q = deque()
    for r in range(h):
        for c in (0, w - 1):
            if bg[r, c] and not reach[r, c]:
                reach[r, c] = True
                q.append((r, c))
    for c in range(w):
        for r in (0, h - 1):
            if bg[r, c] and not reach[r, c]:
                reach[r, c] = True
                q.append((r, c))
    while q:
        r, c = q.popleft()
        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w and bg[nr, nc] and not reach[nr, nc]:
                reach[nr, nc] = True
                q.append((nr, nc))
    return bg & ~reach


def dihedral(i):
    """The 8 dihedral transforms as callables, i in 0..7."""
    def t0(g):
        return g.copy()

    def t1(g):
        return np.rot90(g, 1).copy()

    def t2(g):
        return np.rot90(g, 2).copy()

    def t3(g):
        return np.rot90(g, 3).copy()

    def t4(g):
        return np.fliplr(g).copy()

    def t5(g):
        return np.flipud(g).copy()

    def t6(g):
        return np.transpose(g).copy()

    def t7(g):
        return np.fliplr(np.rot90(g, 1)).copy()
    return [t0, t1, t2, t3, t4, t5, t6, t7][i]


# inverse indices: applying T_k then T_inv(T_k) == identity
DIHEDRAL_INV = {0: 0, 1: 3, 2: 2, 3: 1, 4: 4, 5: 5, 6: 6, 7: 7}


### 2. DSL primitive operations (~90 ops)

In [ ]:
"""DSL primitive operations for ARC program search.
Each op: callable grid -> grid | None (None = not applicable / invalid result).
Ops are deterministic functions of the grid itself (uniform across demos).
Ordered by heuristic priority (most productive first).
"""
import numpy as np


def _ok(g):
    return g is not None and 1 <= g.shape[0] <= MAX_SIDE and 1 <= g.shape[1] <= MAX_SIDE


# ---------------- geometric ----------------
def op_rot90(g):
    return np.rot90(g, 1).copy()


def op_rot180(g):
    return np.rot90(g, 2).copy()


def op_rot270(g):
    return np.rot90(g, 3).copy()


def op_fliplr(g):
    return np.fliplr(g).copy()


def op_flipud(g):
    return np.flipud(g).copy()


def op_transpose(g):
    return np.ascontiguousarray(g.T)


def op_anti_transpose(g):
    return np.ascontiguousarray(np.fliplr(np.rot90(g, 2)).T)


# ---------------- crop ----------------
def op_crop_content(g):
    b = bbox_of(g, None)
    return crop(g, b) if b and (b[2] - b[0] < g.shape[0] or b[3] - b[1] < g.shape[1]) else None


def _mk_crop_color(c):
    def op(g):
        b = bbox_of(g, c)
        if b is None:
            return None
        sub = crop(g, b)
        if sub.shape == g.shape:
            return None
        return sub
    return op


# ---------------- scale ----------------
def _mk_upscale(k):
    def op(g):
        if k <= 1 or g.shape[0] * k > MAX_SIDE or g.shape[1] * k > MAX_SIDE:
            return None
        return np.kron(g, np.ones((k, k), dtype=np.int8))
    return op


def _mk_downscale(k):
    def op(g):
        h, w = g.shape
        if h % k or w % k or k <= 1:
            return None
        bh, bw = h // k, w // k
        blocks = g.reshape(bh, k, bw, k).transpose(0, 2, 1, 3).reshape(bh, bw, -1)
        # mode of each block
        out = np.zeros((bh, bw), dtype=np.int8)
        for i in range(bh):
            for j in range(bw):
                vals, cnts = np.unique(blocks[i, j], return_counts=True)
                out[i, j] = vals[np.argmax(cnts)]
        return out
    return op


# ---------------- tiling ----------------
def _mk_tile(v, hzn):
    def op(g):
        if g.shape[0] * v > MAX_SIDE or g.shape[1] * hzn > MAX_SIDE:
            return None
        return np.tile(g, (v, hzn))
    return op


# ---------------- color ops ----------------
def _mk_recolor(c_from, c_to):
    def op(g):
        if (g == c_from).any():
            out = g.copy()
            out[g == c_from] = c_to
            return out
        return None
    return op


def _mk_bg_to(c):
    def op(g):
        bg = bg_of(g)
        if (g == bg).any():
            out = g.copy()
            out[g == bg] = c
            return out
        return None
    return op


def _mk_del_color(c):
    def op(g):
        bg = bg_of(g)
        if (g == c).any() and c != bg:
            out = g.copy()
            out[g == c] = bg
            return out
        return None
    return op


def _mk_all_nonzero_to(c):
    def op(g):
        if g.dtype != np.int8:
            g = g.astype(np.int8)
        bg = bg_of(g)
        m = g != bg
        if not m.any():
            return None
        out = g.copy()
        out[m] = c
        return out
    return op


def op_set_nonzero_to_dominant_nonzero(g):
    c = dominant_nonbg(g)
    if c is None:
        return None
    bg = bg_of(g)
    out = g.copy()
    out[g != bg] = c
    return out


def op_bg_to_dominant_nonzero(g):
    c = dominant_nonbg(g)
    if c is None:
        return None
    bg = bg_of(g)
    if not (g == bg).any():
        return None
    out = g.copy()
    out[g == bg] = c
    return out


def op_dominant_to_zero(g):
    bg = bg_of(g)
    d = dominant_nonbg(g)
    if d is None:
        return None
    out = g.copy()
    out[g == d] = bg
    return out


def op_color_rank_permute(g):
    """Map colors to frequency-rank order (rank 0 <- most common)."""
    cc = color_counts_local(g)
    order = sorted(cc.items(), key=lambda kv: (-kv[1], kv[0]))
    mapping = {c: i for i, (c, _) in enumerate(order)}
    out = g.copy()
    for c, r in mapping.items():
        out[g == c] = r
    return out


def color_counts_local(g):
    u, c = np.unique(g, return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))


def op_binarily_toggle_bg_dominant(g):
    """Swap background <-> dominant non-bg color."""
    bg = bg_of(g)
    d = dominant_nonbg(g)
    if d is None:
        return None
    out = np.zeros_like(g)
    out[g == d] = bg
    out[g == bg] = d
    rest = (g != d) & (g != bg)
    out[rest] = g[rest]
    return out


# ---------------- object ops ----------------
def op_keep_largest_object(g):
    comps = components(g)
    if len(comps) <= 1:
        return None
    keep = comps[0][1]
    out = np.zeros_like(g)
    for r, c in keep:
        out[r, c] = g[r, c]
    return out


def op_remove_largest_object(g):
    comps = components(g)
    if len(comps) <= 1:
        return None
    drop = comps[0][1]
    out = g.copy()
    for r, c in drop:
        out[r, c] = 0
    return out


def op_keep_largest_per_color(g):
    comps = components(g)
    best = {}
    for col, cells in comps:
        if col not in best or len(cells) > len(best[col]):
            best[col] = cells
    out = np.zeros_like(g)
    for col, cells in best.items():
        for r, c in cells:
            out[r, c] = col
    if (out == g).all():
        return None
    return out


def op_keep_border_touching(g):
    comps = components(g)
    keep = []
    h, w = g.shape
    for col, cells in comps:
        if any(r in (0, h - 1) or c in (0, w - 1) for r, c in cells):
            keep.extend(cells)
    if not keep:
        return None
    out = np.zeros_like(g)
    for r, c in keep:
        out[r, c] = g[r, c]
    return out


def op_remove_border_touching(g):
    comps = components(g)
    drop = []
    h, w = g.shape
    changed = False
    out = g.copy()
    for col, cells in comps:
        if any(r in (0, h - 1) or c in (0, w - 1) for r, c in cells):
            for r, c in cells:
                out[r, c] = 0
            changed = True
    return out if changed else None


def op_fill_enclosed(g):
    mask = flood_enclosed(g)
    if not mask.any():
        return None
    c = dominant_nonzero(g)
    if c is None:
        return None
    out = g.copy()
    out[mask] = c
    return out


def op_fill_enclosed_zero(g):
    """Enclosed background -> dominant nonzero; same as fill_enclosed (kept for clarity)."""
    return op_fill_enclosed(g)


def _mk_gravity(dr, dc):
    """Move every non-background cell as far as possible in (dr,dc) direction."""
    def op(g):
        h, w = g.shape
        bg = bg_of(g)
        out = g.copy()
        moved = False
        if dr != 0:
            for c in range(w):
                col = g[:, c]
                nz = np.where(col != bg)[0]
                if len(nz) == 0:
                    continue
                vals = col[nz]
                if dr > 0:
                    new_pos = h - len(nz) + np.arange(len(nz))
                else:
                    new_pos = np.arange(len(nz))
                if not (new_pos == nz).all():
                    moved = True
                out[new_pos, c] = vals
        else:
            for r in range(h):
                row = g[r, :]
                nz = np.where(row != bg)[0]
                if len(nz) == 0:
                    continue
                vals = row[nz]
                if dc > 0:
                    new_pos = w - len(nz) + np.arange(len(nz))
                else:
                    new_pos = np.arange(len(nz))
                if not (new_pos == nz).all():
                    moved = True
                out[r, new_pos] = vals
        return out if moved else None
    return op


def _mk_extend(dr, dc):
    """Extend every non-background cell to the border along (dr, dc) direction."""
    def op(g):
        h, w = g.shape
        bg = bg_of(g)
        out = g.copy()
        changed = False
        for r in range(h):
            for c in range(w):
                if g[r, c] != bg:
                    rr, cc = r + dr, c + dc
                    while 0 <= rr < h and 0 <= cc < w:
                        if out[rr, cc] != g[r, c]:
                            out[rr, cc] = g[r, c]
                            changed = True
                        rr += dr
                        cc += dc
        return out if changed else None
    return op


# ---------------- symmetry completion ----------------
def _mk_sym_complete(axis, direction):
    """Mirror-complete: axis 0 = horizontal mirror (left<->right), 1 = vertical.
    direction: -1 => fill second half from first half; +1 => fill first half from second half.
    Output same size as input."""
    def op(g):
        h, w = g.shape
        out = g.copy()
        changed = False
        if axis == 0:
            half = w // 2
            if direction < 0:
                src, dst = g[:, :half], np.fliplr(g[:, :half])
                tgt = out[:, w - half:]
                if not (tgt == dst).all():
                    out[:, w - half:] = dst
                    changed = True
            else:
                src = g[:, w - half:]
                dst = np.fliplr(g[:, w - half:])
                tgt = out[:, :half]
                if not (tgt == dst).all():
                    out[:, :half] = dst
                    changed = True
        else:
            half = h // 2
            if direction < 0:
                dst = np.flipud(g[:half, :])
                tgt = out[h - half:, :]
                if not (tgt == dst).all():
                    out[h - half:, :] = dst
                    changed = True
            else:
                dst = np.flipud(g[h - half:, :])
                tgt = out[:half, :]
                if not (tgt == dst).all():
                    out[:half, :] = dst
                    changed = True
        return out if changed else None
    return op


def _mk_sym_append(axis):
    """Append mirrored half: width doubles (or height doubles)."""
    def op(g):
        if axis == 0:
            if g.shape[1] * 2 > MAX_SIDE:
                return None
            return np.hstack([g, np.fliplr(g)])
        if g.shape[0] * 2 > MAX_SIDE:
            return None
        return np.vstack([g, np.flipud(g)])
    return op


# ---------------- noise / misc ----------------
def op_majority_filter(g):
    """3x3 modal filter (denoise)."""
    h, w = g.shape
    out = np.zeros_like(g)
    padded = np.pad(g, 1, mode='constant', constant_values=-1)
    for r in range(h):
        for c in range(w):
            patch = padded[r:r + 3, c:c + 3].flatten()
            vals, cnts = np.unique(patch[patch >= 0], return_counts=True)
            out[r, c] = vals[np.argmax(cnts)]
    if (out == g).all():
        return None
    return out


def op_zero_isolate_border(g):
    """Keep only cells on the border; interior -> background."""
    bg = bg_of(g)
    out = g.copy()
    out[1:-1, 1:-1] = bg
    if (out == g).all():
        return None
    return out


def op_keep_border_only_nonzero(g):
    bg = bg_of(g)
    out = np.full_like(g, bg)
    out[0, :] = g[0, :]
    out[-1, :] = g[-1, :]
    out[:, 0] = g[:, 0]
    out[:, -1] = g[:, -1]
    if (out == g).all() or (out == bg).all():
        return None
    return out


# ---------------- sorting ----------------
def _mk_sort_rows(reverse=False, by_nonzero=False):
    def op(g):
        rows = [tuple(r) for r in g.tolist()]
        if by_nonzero:
            keyed = sorted(rows, key=lambda r: sum(1 for v in r if v != 0), reverse=reverse)
        else:
            keyed = sorted(rows, reverse=reverse)
        out = np.array(keyed, dtype=np.int8).reshape(g.shape)
        if (out == g).all():
            return None
        return out
    return op


def _mk_sort_cols(reverse=False, by_nonzero=False):
    def op(g):
        gt = np.ascontiguousarray(g.T)
        rows = [tuple(r) for r in gt.tolist()]
        if by_nonzero:
            keyed = sorted(rows, key=lambda r: sum(1 for v in r if v != 0), reverse=reverse)
        else:
            keyed = sorted(rows, reverse=reverse)
        out = np.array(keyed, dtype=np.int8).reshape(g.shape).T
        if (out == g).all():
            return None
        return out
    return op


def op_center_content(g):
    """Shift nonzero content bbox to the center of the canvas (same size)."""
    b = bbox_of(g, None)
    if b is None:
        return None
    h, w = g.shape
    ch, cw = b[2] - b[0], b[3] - b[1]
    if (ch, cw) == (h, w):
        return None
    content = crop(g, b)
    res = np.zeros_like(g)
    r0 = (h - ch) // 2
    c0 = (w - cw) // 2
    res[r0:r0 + ch, c0:c0 + cw] = content
    return res


def op_outline_objects(g):
    """Mark background cells adjacent (4-conn) to content with dominant non-bg color."""
    c = dominant_nonbg(g)
    if c is None:
        return None
    h, w = g.shape
    bg = bg_of(g)
    nz = g != bg
    mask = np.zeros_like(g, dtype=bool)
    for r in range(h):
        for cc in range(w):
            if g[r, cc] == bg:
                for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                    nr, nc = r + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and nz[nr, nc]:
                        mask[r, cc] = True
                        break
    if not mask.any():
        return None
    res = g.copy()
    res[mask] = c
    return res


def op_zoom_content(g):
    """Scale the nonzero content bbox by an integer factor to fill the canvas."""
    b = bbox_of(g, None)
    if b is None:
        return None
    h, w = g.shape
    ch, cw = b[2] - b[0], b[3] - b[1]
    if ch == 0 or cw == 0:
        return None
    if h % ch or w % cw:
        return None
    k = h // ch
    if k != w // cw or k < 2 or ch * k > MAX_SIDE or cw * k > MAX_SIDE:
        return None
    content = crop(g, b)
    return np.kron(content, np.ones((k, k), dtype=np.int8))


def op_keep_dominant_color(g):
    """Keep only cells of the most common non-background color."""
    c = dominant_nonbg(g)
    if c is None:
        return None
    bg = bg_of(g)
    out = np.full_like(g, bg)
    out[g == c] = c
    if (out == g).all() or (out == bg).all():
        return None
    return out


def op_content_to_topleft(g):
    """Shift nonzero content bbox to the top-left, keep canvas size."""
    b = bbox_of(g, None)
    if b is None or b[0] == 0 and b[1] == 0:
        return None
    h, w = g.shape
    res = np.zeros_like(g)
    res[:b[2] - b[0], :b[3] - b[1]] = crop(g, b)
    return res


def op_dilate4(g):
    """Grow every non-background cell to its 4-neighbors (same color)."""
    h, w = g.shape
    bg = bg_of(g)
    out = g.copy()
    changed = False
    for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        shifted = np.full_like(g, bg)
        sr = slice(max(0, dr), h + min(0, dr)) if dr else slice(h)
        sc = slice(max(0, dc), w + min(0, dc)) if dc else slice(w)
        tr = slice(max(0, -dr), h + min(0, -dr)) if dr else slice(h)
        tc = slice(max(0, -dc), w + min(0, -dc)) if dc else slice(w)
        shifted[tr, tc] = g[sr, sc]
        grow = (shifted != bg) & (out == bg)
        if grow.any():
            out[grow] = shifted[grow]
            changed = True
    return out if changed else None


def op_fill_bbox_dom(g):
    """Fill the non-background content bbox with the dominant non-bg color."""
    c = dominant_nonbg(g)
    b = bbox_of(g, None)
    if c is None or b is None:
        return None
    out = g.copy()
    out[b[0]:b[2], b[1]:b[3]] = c
    if (out == g).all():
        return None
    return out


def _mk_crop_period(axis):
    """Crop to the minimal repeating period along an axis (shift-equality)."""
    def op(g):
        n = g.shape[axis]
        for p in range(1, n):
            a = g[:, :n - p] if axis == 0 else g[:n - p, :]
            b = g[:, p:] if axis == 0 else g[p:, :]
            if (a == b).all():
                return g[:, :p].copy() if axis == 0 else g[:p, :].copy()
        return None
    return op


def build_ops():
    """Returns list of (name, fn) in priority order."""
    ops = []
    # geometric
    ops += [("rot90", op_rot90), ("rot180", op_rot180), ("rot270", op_rot270),
            ("fliplr", op_fliplr), ("flipud", op_flipud),
            ("transpose", op_transpose), ("anti_transpose", op_anti_transpose)]
    # crop
    ops.append(("crop_content", op_crop_content))
    for c in range(1, 10):
        ops.append((f"crop_color_{c}", _mk_crop_color(c)))
    # scale
    for k in (2, 3, 4):
        ops.append((f"upscale_{k}", _mk_upscale(k)))
    for k in (2, 3):
        ops.append((f"downscale_{k}", _mk_downscale(k)))
    # tile
    for v, hzn in ((2, 2), (3, 3), (1, 2), (2, 1), (1, 3), (3, 1), (2, 3), (3, 2)):
        ops.append((f"tile_{v}x{hzn}", _mk_tile(v, hzn)))
    # symmetry
    ops += [("sym_lr_fill", _mk_sym_complete(0, -1)), ("sym_rl_fill", _mk_sym_complete(0, +1)),
            ("sym_ud_fill", _mk_sym_complete(1, -1)), ("sym_du_fill", _mk_sym_complete(1, +1)),
            ("sym_append_lr", _mk_sym_append(0)), ("sym_append_ud", _mk_sym_append(1))]
    # objects
    ops += [("keep_largest", op_keep_largest_object), ("remove_largest", op_remove_largest_object),
            ("keep_largest_per_color", op_keep_largest_per_color),
            ("keep_border_touch", op_keep_border_touching), ("remove_border_touch", op_remove_border_touching),
            ("fill_enclosed", op_fill_enclosed)]
    # gravity
    ops += [("grav_down", _mk_gravity(1, 0)), ("grav_up", _mk_gravity(-1, 0)),
            ("grav_right", _mk_gravity(0, 1)), ("grav_left", _mk_gravity(0, -1))]
    # extend lines
    ops += [("extend_down", _mk_extend(1, 0)), ("extend_up", _mk_extend(-1, 0)),
            ("extend_right", _mk_extend(0, 1)), ("extend_left", _mk_extend(0, -1))]
    # color
    for c in range(1, 10):
        ops.append((f"nonzero_to_{c}", _mk_all_nonzero_to(c)))
    for c in range(1, 10):
        ops.append((f"zero_to_{c}", _mk_bg_to(c)))
    for c in range(1, 10):
        ops.append((f"del_color_{c}", _mk_del_color(c)))
    ops += [("nonzero_to_dom", op_set_nonzero_to_dominant_nonzero),
            ("bg_to_dom", op_bg_to_dominant_nonzero),
            ("dom_to_zero", op_dominant_to_zero),
            ("swap_bg_dom", op_binarily_toggle_bg_dominant),
            ("color_rank", op_color_rank_permute)]
    # misc
    ops += [("denoise", op_majority_filter), ("zero_interior", op_zero_isolate_border),
            ("border_only", op_keep_border_only_nonzero),
            ("center_content", op_center_content), ("outline", op_outline_objects),
            ("zoom_content", op_zoom_content), ("keep_dom_color", op_keep_dominant_color),
            ("content_topleft", op_content_to_topleft),
            ("dilate4", op_dilate4), ("fill_bbox_dom", op_fill_bbox_dom),
            ("crop_period_rows", _mk_crop_period(0)), ("crop_period_cols", _mk_crop_period(1))]
    # sorting
    ops += [("sort_rows_asc", _mk_sort_rows(False)), ("sort_rows_desc", _mk_sort_rows(True)),
            ("sort_cols_asc", _mk_sort_cols(False)), ("sort_cols_desc", _mk_sort_cols(True)),
            ("sort_rows_nz", _mk_sort_rows(False, True)), ("sort_cols_nz", _mk_sort_cols(False, True))]
    return ops


OPS = build_ops()
OP_INDEX = {name: i for i, (name, _) in enumerate(OPS)}

# priority order: geometric/crop/scale/tile/sym first (cheap & common), then objects, colors
OP_PRIORITY = [i for i, (n, _) in enumerate(OPS)]


### 3. Beam-search program synthesis

In [ ]:
"""Beam-search program synthesis over the DSL.
A program is a list of op names. It must map every demo input to its demo output
exactly. Found programs are then applied to the test inputs.
"""
import time
import numpy as np

MISMATCH_SIZE_PENALTY = 400


def mismatch(g, target):
    """Distance between produced grid and target demo output."""
    if g.shape != target.shape:
        return (MISMATCH_SIZE_PENALTY + 20 * (abs(g.shape[0] - target.shape[0]) +
                                             abs(g.shape[1] - target.shape[1]))
                + min(g.size, target.size))
    return int((g != target).sum())


def apply_program(prog_names, grid):
    """Apply a program (list of op names) to a grid. Returns grid or None."""
    g = grid
    for name in prog_names:
        fn = OPS_DICT.get(name)
        if fn is None:
            return None
        try:
            g = fn(g)
        except Exception:
            return None
        if g is None or g.shape[0] > MAX_SIDE or g.shape[1] > MAX_SIDE or g.shape[0] < 1 or g.shape[1] < 1:
            return None
    return g


OPS_DICT = {name: fn for name, fn in OPS}


def _demo_state_hash(grids):
    return hash(tuple((g.shape, g.tobytes()) for g in grids))


def beam_search(demos, deadline, beam_width=220, max_depth=6, op_order=None,
                collect_size=3, verbose=False):
    """Search for op-sequences mapping demo inputs -> demo outputs.

    demos: list of (input_grid, output_grid)
    Returns (full_solutions, partials):
      full_solutions: list of programs solving ALL demos (shortest first)
      partials: list of (program, exact_demo_count, total_mismatch) without full solve,
                best first, max 3 entries
    """
    n = len(demos)
    inputs = [d[0] for d in demos]
    targets = [d[1] for d in demos]

    # states: list of (score_sum, program_tuple, grids_tuple)
    start_score = sum(mismatch(g, t) for g, t in zip(inputs, targets))
    beam = [(start_score, (), tuple(inputs))]
    seen = {_demo_state_hash(inputs)}

    full_solutions = []
    partials = []  # (program, exact_demo_count, total_mismatch)
    start_vecs0 = tuple(mismatch(g, t) for g, t in zip(inputs, targets))
    start_exact = sum(1 for v in start_vecs0 if v == 0)

    op_list = [(i, name, fn) for i, (name, fn) in enumerate(OPS)]
    if op_order is not None:
        op_list = [op_list[i] for i in op_order]

    for depth in range(max_depth):
        if time.monotonic() > deadline or not beam:
            break
        candidates = []
        for score_sum, prog, grids in beam:
            if score_sum == 0:
                continue  # already solved; don't expand
            for _, name, fn in op_list:
                new_grids = []
                ok = True
                for g in grids:
                    try:
                        ng = fn(g)
                    except Exception:
                        ng = None
                    if ng is None:
                        ok = False
                        break
                    new_grids.append(ng)
                if not ok:
                    continue
                h = _demo_state_hash(new_grids)
                if h in seen:
                    continue
                seen.add(h)
                new_vec = tuple(mismatch(g, t) for g, t in zip(new_grids, targets))
                candidates.append((sum(new_vec), prog + (name,), tuple(new_grids), new_vec))
                if time.monotonic() > deadline:
                    break
            if time.monotonic() > deadline:
                break

        if not candidates:
            break

        candidates.sort(key=lambda x: x[0])
        for s, prog, grids, vec in candidates:
            if s == 0:
                if prog not in full_solutions:
                    full_solutions.append(prog)
            else:
                exact_cnt = sum(1 for v in vec if v == 0)
                if exact_cnt > 0:
                    partials.append((prog, exact_cnt, s))

        if full_solutions:
            break  # shallowest solutions preferred

        beam = [(s, prog, grids) for s, prog, grids, _ in candidates[:beam_width]]
        # keep also the untouched start state's solved children? beam prune done

        if verbose:
            print(f"  depth {depth + 1}: best={candidates[0][0]} beam={len(beam)}")

    # also record start-level partial if nothing found
    if not partials and start_exact > 0:
        partials.append((None, start_exact, sum(start_vecs0)))

    if len(full_solutions) > collect_size:
        full_solutions = full_solutions[:collect_size]
    # dedupe + rank partials: most exact demos first, then least mismatch
    uniq = {}
    for prog, ex, tot in partials:
        if prog not in uniq or tot < uniq[prog][1]:
            uniq[prog] = (ex, tot)
    partials = sorted(uniq.items(), key=lambda kv: (-kv[1][0], kv[1][1]))
    partials = [(p, ex, tot) for p, (ex, tot) in partials][:3]
    return full_solutions, partials


### 4. Verified heuristic solver bank

In [ ]:
"""Verified heuristic solver bank.
Each heuristic fits parameters on the demonstration pairs and is returned ONLY
if it reproduces every demo output exactly (verification step). This guarantees
no false-positive predictions from buggy heuristics.

API: heuristics_solve(demos, deadline) -> list of (name, predict_fn) verified,
sorted by a confidence prior (more specific rules first).
predict_fn(grid) -> grid or None.
"""
import numpy as np

DIRECTIONS = {"down": (1, 0), "up": (-1, 0), "right": (0, 1), "left": (0, -1)}


# ---------------- helpers ----------------
def _same(g1, g2):
    return g1 is not None and g2 is not None and g1.shape == g2.shape and bool((g1 == g2).all())


def _apply_all(fn, demos):
    """Apply fn to all demo inputs; returns list or None on failure."""
    outs = []
    for gin, gout in demos:
        try:
            g = fn(gin)
        except Exception:
            return None
        if g is None or g.shape[0] > MAX_SIDE or g.shape[1] > MAX_SIDE:
            return None
        outs.append(g)
    return outs


def _verified(fn, demos):
    outs = _apply_all(fn, demos)
    if outs is None:
        return None
    for g, (_, gout) in zip(outs, demos):
        if not _same(g, gout):
            return None
    return True


def _mk_fn(base_fn, demos, wrap=None):
    """Fit helper: returns predict_fn if base_fn reproduces all demos."""
    if _verified(base_fn, demos):
        return base_fn
    return None


# ---------------- individual heuristic builders ----------------
def h_dihedral(demos):
    out = []
    for i in range(8):
        t = dihedral(i)
        name = ["id", "rot90", "rot180", "rot270", "fliplr", "flipud", "transpose", "anti_tr"][i]
        fn = t
        if _verified(fn, demos):
            out.append((name if name != "id" else "identity", fn, 0.5))
    return out


def h_recolor_map(demos):
    """Position-wise color map (requires equal shapes). Consistent across demos."""
    mapping = {}

    def fit_map(demos):
        m = {}
        for gin, gout in demos:
            if gin.shape != gout.shape:
                return None
            pairs = zip(gin.flatten().tolist(), gout.flatten().tolist())
            for a, b in pairs:
                if a in m:
                    if m[a] != b:
                        return None
                else:
                    m[a] = b
        return m

    m = fit_map(demos)
    if m is None:
        return []

    def fn(g):
        out = np.zeros_like(g)
        for r in range(g.shape[0]):
            for c in range(g.shape[1]):
                v = int(g[r, c])
                out[r, c] = m.get(v, 0)
        return out

    # fn must reproduce demos (it does by construction) - verify anyway
    if _verified(fn, demos):
        return [("recolor_map", fn, 0.8)]
    return []


def h_crop(demos):
    out = []
    # crop to nonzero bbox
    def fn_content(g):
        b = bbox_of(g, None)
        if b is None:
            return None
        return crop(g, b)
    if _verified(fn_content, demos):
        out.append(("crop_content", fn_content, 0.9))
    # crop to bbox of specific color
    for c in range(1, 10):
        def mk(c):
            def fn(g):
                b = bbox_of(g, c)
                if b is None:
                    return None
                return crop(g, b)
            return fn
        fn = mk(c)
        if _verified(fn, demos):
            out.append((f"crop_color_{c}", fn, 0.9))
    # crop with margin 1 (content bbox grown by 1, clipped)
    def fn_content_m1(g):
        b = bbox_of(g, None)
        if b is None:
            return None
        b2 = (max(0, b[0] - 1), max(0, b[1] - 1), min(g.shape[0], b[2] + 1), min(g.shape[1], b[3] + 1))
        if b2 == (0, 0, g.shape[0], g.shape[1]):
            return None
        return crop(g, b2)
    if _verified(fn_content_m1, demos):
        out.append(("crop_content_m1", fn_content_m1, 0.9))
    return out


def h_tile(demos):
    for v in range(1, 6):
        for hzn in range(1, 6):
            if v == 1 and hzn == 1:
                continue
            def mk(v, hzn):
                def fn(g):
                    if g.shape[0] * v > MAX_SIDE or g.shape[1] * hzn > MAX_SIDE:
                        return None
                    return np.tile(g, (v, hzn))
                return fn
            fn = mk(v, hzn)
            if _verified(fn, demos):
                return [(f"tile_{v}x{hzn}", fn, 0.9)]
    return []


def h_scale(demos):
    out = []
    for k in range(2, 6):
        def mkup(k):
            def fn(g):
                if g.shape[0] * k > MAX_SIDE or g.shape[1] * k > MAX_SIDE:
                    return None
                return np.kron(g, np.ones((k, k), dtype=np.int8))
            return fn
        fn = mkup(k)
        if _verified(fn, demos):
            out.append((f"upscale_{k}", fn, 0.9))
    for k in (2, 3, 4):
        def mkdn(k):
            def fn(g):
                h, w = g.shape
                if h % k or w % k:
                    return None
                bh, bw = h // k, w // k
                blocks = g.reshape(bh, k, bw, k).transpose(0, 2, 1, 3).reshape(bh, bw, -1)
                res = np.zeros((bh, bw), dtype=np.int8)
                for i in range(bh):
                    for j in range(bw):
                        vals, cnts = np.unique(blocks[i, j], return_counts=True)
                        res[i, j] = vals[np.argmax(cnts)]
                return res
            return fn
        fn = mkdn(k)
        if _verified(fn, demos):
            out.append((f"downscale_{k}", fn, 0.9))
    return out


def h_sym_completion(demos):
    out = []
    for axis, aname in ((0, "lr"), (1, "ud")):
        for direction, dname in ((-1, "fill2"), (1, "fill1")):
            def mk(axis, direction):
                def fn(g):
                    h, w = g.shape
                    res = g.copy()
                    if axis == 0:
                        half = w // 2
                        if direction < 0:
                            res[:, w - half:] = np.fliplr(g[:, :half])
                        else:
                            res[:, :half] = np.fliplr(g[:, w - half:])
                    else:
                        half = h // 2
                        if direction < 0:
                            res[h - half:, :] = np.flipud(g[:half, :])
                        else:
                            res[:half, :] = np.flipud(g[h - half:, :])
                    return res
                return fn
            fn = mk(axis, direction)
            if _verified(fn, demos):
                out.append((f"sym_{aname}_{dname}", fn, 0.9))
    for axis, aname in ((0, "lr"), (1, "ud")):
        def mk(axis):
            def fn(g):
                if axis == 0:
                    if g.shape[1] * 2 > MAX_SIDE:
                        return None
                    return np.hstack([g, np.fliplr(g)])
                if g.shape[0] * 2 > MAX_SIDE:
                    return None
                return np.vstack([g, np.flipud(g)])
            return fn
        fn = mk(axis)
        if _verified(fn, demos):
            out.append((f"sym_append_{aname}", fn, 0.9))
    return out


def h_fill_enclosed(demos):
    """Fill enclosed background with a consistent color."""
    for c in range(1, 10):
        def mk(c):
            def fn(g):
                mask = flood_enclosed(g)
                if not mask.any():
                    return None
                res = g.copy()
                res[mask] = c
                return res
            return fn
        fn = mk(c)
        if _verified(fn, demos):
            return [(f"fill_enclosed_{c}", fn, 0.9)]
    return []


def h_object_filter(demos):
    out = []
    cands = {
        "keep_largest": lambda g: _keep_n_largest(g, 1),
        "keep_2_largest": lambda g: _keep_n_largest(g, 2),
        "remove_largest": lambda g: _remove_n_largest(g, 1),
    }
    for name, fn in cands.items():
        if _verified(fn, demos):
            out.append((name, fn, 0.85))
    return out


def _keep_n_largest(g, n):
    comps = components(g)
    if len(comps) < n:
        return None
    res = np.zeros_like(g)
    for col, cells in comps[:n]:
        for r, c in cells:
            res[r, c] = col
    if (res == g).all():
        return None
    return res


def _remove_n_largest(g, n):
    comps = components(g)
    if len(comps) <= n:
        return None
    res = g.copy()
    for col, cells in comps[:n]:
        for r, c in cells:
            res[r, c] = 0
    return res


def h_gravity(demos):
    out = []
    for name, (dr, dc) in DIRECTIONS.items():
        def mk(dr, dc):
            def fn(g):
                h, w = g.shape
                bg = bg_of(g)
                res = g.copy()
                moved = False
                if dr != 0:
                    for c in range(w):
                        nz = np.where(g[:, c] != bg)[0]
                        if len(nz) == 0:
                            continue
                        vals = g[nz, c]
                        newp = h - len(nz) + np.arange(len(nz)) if dr > 0 else np.arange(len(nz))
                        if not (newp == nz).all():
                            moved = True
                        res[newp, c] = vals
                else:
                    for r in range(h):
                        nz = np.where(g[r, :] != bg)[0]
                        if len(nz) == 0:
                            continue
                        vals = g[r, nz]
                        newp = w - len(nz) + np.arange(len(nz)) if dc > 0 else np.arange(len(nz))
                        if not (newp == nz).all():
                            moved = True
                        res[r, newp] = vals
                return res if moved else None
            return fn
        fn = mk(dr, dc)
        if _verified(fn, demos):
            out.append((f"gravity_{name}", fn, 0.85))
    return out


def h_extend(demos):
    out = []
    for name, (dr, dc) in DIRECTIONS.items():
        def mk(dr, dc):
            def fn(g):
                h, w = g.shape
                bg = bg_of(g)
                res = g.copy()
                changed = False
                for r in range(h):
                    for c in range(w):
                        if g[r, c] != bg:
                            rr, cc = r + dr, c + dc
                            while 0 <= rr < h and 0 <= cc < w:
                                if res[rr, cc] != g[r, c]:
                                    res[rr, cc] = g[r, c]
                                    changed = True
                                rr += dr
                                cc += dc
                return res if changed else None
            return fn
        fn = mk(dr, dc)
        if _verified(fn, demos):
            out.append((f"extend_{name}", fn, 0.85))
    return out


def h_frame(demos):
    """Draw a border of color c over the input."""
    for c in range(1, 10):
        def mk(c):
            def fn(g):
                res = g.copy()
                res[0, :] = c
                res[-1, :] = c
                res[:, 0] = c
                res[:, -1] = c
                return res
            return fn
        fn = mk(c)
        if _verified(fn, demos):
            return [(f"frame_{c}", fn, 0.85)]
    return []


def h_object_size_color(demos):
    """Recolor each object by its size (consistent size->color map)."""
    m = {}
    for gin, gout in demos:
        if gin.shape != gout.shape:
            return []
        comps_in = {(min(p[0] for p in cells), min(p[1] for p in cells)): (col, len(cells))
                    for col, cells in components(gin)}
        changed = False
        for col, cells in components(gout):
            key = (min(p[0] for p in cells), min(p[1] for p in cells))
            if key in comps_in:
                _, sz = comps_in[key]
                oc = int(col)
                if sz in m and m[sz] != oc:
                    return []
                m[sz] = oc
                changed = True
    if not m:
        return []

    def fn(g):
        res = g.copy()
        for col, cells in components(g):
            sz = len(cells)
            if sz in m:
                for r, c in cells:
                    res[r, c] = m[sz]
        return res

    if _verified(fn, demos):
        return [("objsize_recolor", fn, 0.85)]
    return []


def h_move_object(demos):
    """Single content block moved by a consistent rule (constant delta or corner snap)."""
    deltas = []
    for gin, gout in demos:
        bi = bbox_of(gin, None)
        bo = bbox_of(gout, None)
        if bi is None or bo is None:
            return []
        if (bi[2] - bi[0], bi[3] - bi[1]) != (bo[2] - bo[0], bo[3] - bo[1]):
            return []
        content = crop(gin, bi)
        if not _same(crop(gout, bo), content):
            return []
        deltas.append((bo[0] - bi[0], bo[1] - bi[1]))
    if not deltas:
        return []
    # constant delta?
    if len(set(deltas)) == 1:
        dr, dc = deltas[0]

        def fn(g):
            b = bbox_of(g, None)
            if b is None:
                return None
            content = crop(g, b)
            h, w = b[2] - b[0] + dr, b[3] - b[1] + dc
            if h > MAX_SIDE or w > MAX_SIDE or h < 1 or w < 1:
                return None
            res = np.zeros((h, w), dtype=np.int8)
            r0 = b[0] + dr
            c0 = b[1] + dc
            if r0 < 0 or c0 < 0 or r0 + content.shape[0] > h or c0 + content.shape[1] > w:
                return None
            res[r0:r0 + content.shape[0], c0:c0 + content.shape[1]] = content
            return res
        if _verified(fn, demos):
            return [("move_const", fn, 0.85)]
    return []


def h_keep_color(demos):
    """Keep only cells of color c (others -> background)."""
    for c in range(1, 10):
        def mk(c):
            def fn(g):
                if not (g == c).any():
                    return None
                bg = bg_of(g)
                res = np.full_like(g, bg)
                res[g == c] = c
                return res
            return fn
        fn = mk(c)
        if _verified(fn, demos):
            return [(f"keep_color_{c}", fn, 0.85)]
    return []


def h_recolor_scalar(demos):
    """Single color->color replacement (consistent map, equal shapes optional)."""
    for a in range(0, 10):
        for b in range(0, 10):
            if a == b:
                continue
            def mk(a, b):
                def fn(g):
                    if not (g == a).any():
                        return None
                    res = g.copy()
                    res[g == a] = b
                    return res
                return fn
            fn = mk(a, b)
            if _verified(fn, demos):
                return [(f"recolor_{a}_{b}", fn, 0.8)]
    return []


def h_pad_center(demos):
    """Pad content centered to fixed demo output size."""
    sizes = {(gout.shape) for _, gout in demos}
    if len(sizes) != 1:
        return []
    H, W = list(sizes)[0]

    def fn(g):
        b = bbox_of(g, None)
        if b is None:
            return None
        content = crop(g, b)
        if content.shape[0] > H or content.shape[1] > W:
            return None
        res = np.zeros((H, W), dtype=np.int8)
        r0 = (H - content.shape[0]) // 2
        c0 = (W - content.shape[1]) // 2
        res[r0:r0 + content.shape[0], c0:c0 + content.shape[1]] = content
        return res
    if _verified(fn, demos):
        return [("pad_center", fn, 0.85)]
    return []


def h_sort(demos):
    out = []
    variants = {
        "sort_rows_asc": lambda g: _sort_rows(g, False, False),
        "sort_rows_desc": lambda g: _sort_rows(g, True, False),
        "sort_cols_asc": lambda g: _sort_cols(g, False, False),
        "sort_cols_desc": lambda g: _sort_cols(g, True, False),
        "sort_rows_nz": lambda g: _sort_rows(g, False, True),
        "sort_cols_nz": lambda g: _sort_cols(g, False, True),
    }
    for name, fn in variants.items():
        if _verified(fn, demos):
            out.append((name, fn, 0.85))
    return out


def _sort_rows(g, reverse, by_nonzero):
    rows = [tuple(r) for r in g.tolist()]
    if by_nonzero:
        keyed = sorted(rows, key=lambda r: sum(1 for v in r if v != 0), reverse=reverse)
    else:
        keyed = sorted(rows, reverse=reverse)
    return np.array(keyed, dtype=np.int8).reshape(g.shape)


def _sort_cols(g, reverse, by_nonzero):
    gt = np.ascontiguousarray(g.T)
    rows = [tuple(r) for r in gt.tolist()]
    if by_nonzero:
        keyed = sorted(rows, key=lambda r: sum(1 for v in r if v != 0), reverse=reverse)
    else:
        keyed = sorted(rows, reverse=reverse)
    return np.array(keyed, dtype=np.int8).reshape(g.shape).T


def h_outline(demos):
    """Mark background cells adjacent to content with color c."""
    for c in range(1, 10):
        def mk(c):
            def fn(g):
                h, w = g.shape
                bg = bg_of(g)
                nz = g != bg
                mask = np.zeros_like(g, dtype=bool)
                for r in range(h):
                    for cc in range(w):
                        if g[r, cc] == bg:
                            for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
                                nr, nc = r + dr, cc + dc
                                if 0 <= nr < h and 0 <= nc < w and nz[nr, nc]:
                                    mask[r, cc] = True
                                    break
                if not mask.any():
                    return None
                res = g.copy()
                res[mask] = c
                return res
            return fn
        fn = mk(c)
        if _verified(fn, demos):
            return [(f"outline_{c}", fn, 0.85)]
    return []


def h_recolor_by_position(demos):
    """Recolor objects by the row (or column) band of their bbox: consistent map."""
    for axis, aname in ((0, "row"), (1, "col")):
        m = {}
        ok = True
        for gin, gout in demos:
            if gin.shape != gout.shape:
                ok = False
                break
            comp_in = {(min(p[0] for p in cells), min(p[1] for p in cells)): col
                       for col, cells in components(gin)}
            for col, cells in components(gout):
                key = (min(p[0] for p in cells), min(p[1] for p in cells))
                if key in comp_in:
                    band = key[0] if axis == 0 else key[1]
                    oc = int(col)
                    if band in m and m[band] != oc:
                        ok = False
                        break
                    m[band] = oc
            if not ok:
                break
        if not ok or not m:
            continue

        def mk(m, axis):
            def fn(g):
                res = g.copy()
                for col, cells in components(g):
                    band = min(p[0] for p in cells) if axis == 0 else min(p[1] for p in cells)
                    if band in m:
                        for r, c in cells:
                            res[r, c] = m[band]
                return res
            return fn
        fn = mk(dict(m), axis)
        if _verified(fn, demos):
            return [(f"objpos_recolor_{aname}", fn, 0.85)]
    return []


def h_center(demos):
    def fn(g):
        b = bbox_of(g, None)
        if b is None:
            return None
        h, w = g.shape
        ch, cw = b[2] - b[0], b[3] - b[1]
        if (ch, cw) == (h, w):
            return None
        res = np.zeros_like(g)
        r0 = (h - ch) // 2
        c0 = (w - cw) // 2
        res[r0:r0 + ch, c0:c0 + cw] = crop(g, b)
        return res
    if _verified(fn, demos):
        return [("center_content", fn, 0.85)]
    return []


def h_zoom(demos):
    def fn(g):
        b = bbox_of(g, None)
        if b is None:
            return None
        h, w = g.shape
        ch, cw = b[2] - b[0], b[3] - b[1]
        if ch == 0 or cw == 0 or h % ch or w % cw:
            return None
        k = h // ch
        if k != w // cw or k < 2 or ch * k > MAX_SIDE or cw * k > MAX_SIDE:
            return None
        return np.kron(crop(g, b), np.ones((k, k), dtype=np.int8))
    if _verified(fn, demos):
        return [("zoom_content", fn, 0.9)]
    return []


def h_topleft(demos):
    def fn(g):
        b = bbox_of(g, None)
        if b is None or (b[0] == 0 and b[1] == 0):
            return None
        res = np.zeros_like(g)
        res[:b[2] - b[0], :b[3] - b[1]] = crop(g, b)
        return res
    if _verified(fn, demos):
        return [("content_topleft", fn, 0.85)]
    return []


def h_dilate(demos):
    def fn(g):
        h, w = g.shape
        bg = bg_of(g)
        out = g.copy()
        changed = False
        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
            shifted = np.full_like(g, bg)
            sr = slice(max(0, dr), h + min(0, dr)) if dr else slice(h)
            sc = slice(max(0, dc), w + min(0, dc)) if dc else slice(w)
            tr = slice(max(0, -dr), h + min(0, -dr)) if dr else slice(h)
            tc = slice(max(0, -dc), w + min(0, -dc)) if dc else slice(w)
            shifted[tr, tc] = g[sr, sc]
            grow = (shifted != bg) & (out == bg)
            if grow.any():
                out[grow] = shifted[grow]
                changed = True
        return out if changed else None
    if _verified(fn, demos):
        return [("dilate4", fn, 0.85)]
    return []


def h_modal_window(demos):
    """Pattern-in-noise: output = the most frequently repeated sub-window of the
    input, with the window size taken from the demo output size (count >= 2,
    deterministic tie-break by first appearance)."""
    sizes = {gout.shape for _, gout in demos}
    if len(sizes) != 1:
        return []
    H, W = list(sizes)[0]
    if H > 10 or W > 10 or H * W < 2:
        return []

    def modal(gin):
        h, w = gin.shape
        if h < H or w < W or (h == H and w == W):
            return None
        counts = {}
        first = {}
        for r in range(h - H + 1):
            for c in range(w - W + 1):
                win = gin[r:r + H, c:c + W]
                key = win.tobytes()
                counts[key] = counts.get(key, 0) + 1
                if key not in first:
                    first[key] = (r, c, win.copy())
        # candidates with count >= 2; tie-break: higher count, then earliest appearance
        cand = [(cnt, -first[k][0], -first[k][1], first[k][2])
                for k, cnt in counts.items() if cnt >= 2]
        if not cand:
            return None
        cand.sort(key=lambda x: (-x[0], -x[1], -x[2]))
        return cand[0][3]

    def fn(g):
        m = modal(g)
        if m is None:
            return None
        return m.copy()
    if _verified(fn, demos):
        return [("modal_window", fn, 0.9)]
    return []


def h_translate_all(demos):
    """Translate ALL content by a constant delta, same canvas size."""
    deltas = []
    for gin, gout in demos:
        if gin.shape != gout.shape:
            return []
        bi = bbox_of(gin, None)
        bo = bbox_of(gout, None)
        if bi is None or bo is None:
            return []
        if (bi[2] - bi[0], bi[3] - bi[1]) != (bo[2] - bo[0], bo[3] - bo[1]):
            return []
        if not _same(crop(gin, bi), crop(gout, bo)):
            return []
        if bi == bo:
            return []
        deltas.append((bo[0] - bi[0], bo[1] - bi[1]))
    if not deltas or len(set(deltas)) != 1:
        return []
    dr, dc = deltas[0]

    def fn(g):
        b = bbox_of(g, None)
        if b is None:
            return None
        res = np.zeros_like(g)
        r0, c0 = b[0] + dr, b[1] + dc
        ch, cw = b[2] - b[0], b[3] - b[1]
        if r0 < 0 or c0 < 0 or r0 + ch > g.shape[0] or c0 + cw > g.shape[1]:
            return None
        res[r0:r0 + ch, c0:c0 + cw] = crop(g, b)
        return res
    if _verified(fn, demos):
        return [("translate_all", fn, 0.85)]
    return []


HEURISTIC_BUILDERS = [
    (h_dihedral, 3),
    (h_recolor_map, 5),
    (h_crop, 5),
    (h_tile, 4),
    (h_scale, 4),
    (h_sym_completion, 4),
    (h_fill_enclosed, 3),
    (h_object_filter, 3),
    (h_gravity, 3),
    (h_extend, 2),
    (h_frame, 3),
    (h_object_size_color, 4),
    (h_move_object, 4),
    (h_keep_color, 3),
    (h_recolor_scalar, 2),
    (h_pad_center, 3),
    (h_sort, 3),
    (h_outline, 3),
    (h_recolor_by_position, 4),
    (h_center, 3),
    (h_zoom, 4),
    (h_topleft, 3),
    (h_dilate, 3),
    (h_translate_all, 3),
    (h_modal_window, 5),
]


def heuristics_solve(demos, deadline):
    """Run all heuristic builders; return verified ones sorted by confidence desc."""
    if not demos:
        return []
    out = []
    for builder, conf in HEURISTIC_BUILDERS:
        if time_over(deadline):
            break
        try:
            res = builder(demos)
        except Exception:
            res = []
        out.extend(res)
    out.sort(key=lambda x: -x[2])
    return out


def time_over(deadline):
    import time as _t
    return _t.monotonic() > deadline


### 5. Prediction pipeline & time-budgeted run loop

In [ ]:
"""Main prediction pipeline for ARC Prize 2026 (ARC-AGI-2).

Strategy per task (all verified against demonstration pairs, never against
test outputs - no leakage):
  1. Verified heuristic bank (fast, parameter-free rules fitted on demos)
  2. DSL beam-search program synthesis on demo pairs
  3. Same search under the 8 dihedral augmentations (with inverse transform back)
  4. Partial-match programs / priors as second attempt
  5. Always output valid grids: 2 attempts per test input.
"""
import json
import os
import time
import numpy as np


DEFAULT_CFG = {
    "heur_deadline": 2.0,        # seconds for heuristic bank
    "dsl_deadline": 40.0,        # seconds for non-augmented DSL search
    "aug_deadline": 30.0,        # seconds total for augmented search
    "beam_width": 220,
    "aug_beam_width": 120,
    "max_depth": 6,
    "total_budget": 8.0 * 3600,  # wall-clock budget for the whole run
    "safety_margin": 0.06,       # fraction of budget kept as safety
    "progress_every": 10,
}


def _distinct(cands):
    """Filter candidate grids to distinct (shape, bytes)."""
    seen = set()
    out = []
    for g in cands:
        if g is None:
            continue
        key = (g.shape, g.tobytes())
        if key in seen:
            continue
        seen.add(key)
        out.append(g)
    return out


def _valid(g):
    return (g is not None and 1 <= g.shape[0] <= MAX_SIDE and 1 <= g.shape[1] <= MAX_SIDE
            and g.min() >= 0 and g.max() <= 9)


def infer_fallbacks(demos, test_grid):
    """Blind prior candidates when nothing verified."""
    cands = []
    if test_grid is None:
        return cands
    cands.append(test_grid.copy())  # identity
    b = bbox_of(test_grid, None)
    if b is not None and (b[2] - b[0] < test_grid.shape[0] or b[3] - b[1] < test_grid.shape[1]):
        cands.append(crop(test_grid, b))
    if demos:
        # consistent integer upscale ratio
        ratios = set()
        for gin, gout in demos:
            if gout.shape[0] % gin.shape[0] == 0 and gout.shape[1] % gin.shape[1] == 0 \
                    and gout.shape[0] // gin.shape[0] == gout.shape[1] // gin.shape[1] \
                    and gout.shape[0] // gin.shape[0] >= 2:
                ratios.add(gout.shape[0] // gin.shape[0])
        if len(ratios) == 1:
            k = ratios.pop()
            if test_grid.shape[0] * k <= MAX_SIDE and test_grid.shape[1] * k <= MAX_SIDE:
                cands.append(np.kron(test_grid, np.ones((k, k), dtype=np.int8)))
        # consistent tile ratio
        tratios = set()
        for gin, gout in demos:
            if gout.shape[0] % gin.shape[0] == 0 and gout.shape[1] % gin.shape[1] == 0:
                tratios.add((gout.shape[0] // gin.shape[0], gout.shape[1] // gin.shape[1]))
        if len(tratios) == 1:
            v, hz = tratios.pop()
            if test_grid.shape[0] * v <= MAX_SIDE and test_grid.shape[1] * hz <= MAX_SIDE \
                    and (v > 1 or hz > 1):
                cands.append(np.tile(test_grid, (v, hz)))
        # fixed demo output size (top-left pad)
        sizes = {gout.shape for _, gout in demos}
        if len(sizes) == 1:
            H, W = list(sizes)[0]
            g = np.zeros((H, W), dtype=np.int8)
            hh = min(test_grid.shape[0], H)
            ww = min(test_grid.shape[1], W)
            g[:hh, :ww] = test_grid[:hh, :ww]
            cands.append(g)
        # position-wise recolor map (if sizes equal in demos)
        m = {}
        ok = True
        for gin, gout in demos:
            if gin.shape != gout.shape:
                ok = False
                break
            for a, bb in zip(gin.flatten().tolist(), gout.flatten().tolist()):
                if a in m and m[a] != bb:
                    ok = False
                    break
                m[a] = bb
            if not ok:
                break
        if ok:
            res = np.zeros_like(test_grid)
            for r in range(test_grid.shape[0]):
                for c in range(test_grid.shape[1]):
                    res[r, c] = m.get(int(test_grid[r, c]), 0)
            cands.append(res)
    return _distinct([c for c in cands if _valid(c)])


def solve_task(task_dict, cfg, deadline_task, return_tags=False):
    """task_dict: {'train': [(in_list, out_list), ...], 'test': [in_list, ...]}
    Returns list (per test input) of up to 2 candidate output grids (numpy).
    If return_tags, each candidate is (source_tag, grid)."""
    demos = []
    for p in task_dict.get("train", []):
        if isinstance(p, dict):
            gi = to_grid(p.get("input"))
            go = to_grid(p.get("output"))
        else:
            gi = to_grid(p[0])
            go = to_grid(p[1])
        if gi is not None and go is not None:
            demos.append((gi, go))
    tests = [to_grid(p[0] if isinstance(p, (list, tuple)) and len(p) == 1 else p)
             for p in task_dict.get("test", [])]
    # 'test' entries may be raw grids OR dicts with 'input'
    tests = []
    for p in task_dict.get("test", []):
        if isinstance(p, dict):
            tests.append(to_grid(p.get("input")))
        else:
            tests.append(to_grid(p))

    per_test = [[] for _ in tests]
    if not tests:
        return per_test

    heur_preds = []
    full_progs = []
    partial_progs = []
    aug_progs = []

    now = time.monotonic()
    if demos and now < deadline_task:
        try:
            heur_preds = heuristics_solve(demos, min(deadline_task, now + cfg["heur_deadline"]))
        except Exception:
            heur_preds = []

    now = time.monotonic()
    if demos and now < deadline_task:
        try:
            full_progs, partials = beam_search(
                demos,
                deadline=min(deadline_task, now + cfg["dsl_deadline"]),
                beam_width=cfg["beam_width"],
                max_depth=cfg["max_depth"])
            if partials and not full_progs:
                partial_progs = [p for p, ex, tot in partials if p]
        except Exception:
            full_progs = []

    now = time.monotonic()
    if demos and not full_progs and now < deadline_task:
        # dihedral augmentation (skip identity - already covered)
        remaining = min(deadline_task, now + cfg["aug_deadline"]) - now
        per_t = max(2.0, remaining / 7.0)
        for k in (1, 2, 3, 4, 5, 6, 7):
            t = time.monotonic()
            if t >= deadline_task or per_t < 2.0:
                break
            T = dihedral(k)
            Tinv = dihedral(DIHEDRAL_INV[k])
            aug_demos = [(T(gi), T(go)) for gi, go in demos]
            try:
                progs, _ = beam_search(
                    aug_demos,
                    deadline=t + per_t,
                    beam_width=cfg["aug_beam_width"],
                    max_depth=cfg["max_depth"])
            except Exception:
                progs = []
            if progs:
                aug_progs.append((k, progs[0]))

    # ---------------- assemble candidates per test input ----------------
    for i, tg in enumerate(tests):
        if tg is None:
            per_test[i] = []
            continue
        prio = []
        tags = {}
        # 1. verified heuristics
        for name, fn, conf in heur_preds:
            try:
                g = fn(tg)
            except Exception:
                g = None
            if _valid(g):
                prio.append((1.0, g))
                tags.setdefault(_key(g), "heur:" + name.split(":")[0].split("_")[0])
        # 2. full DSL solutions
        for prog in full_progs:
            g = apply_program(prog, tg)
            if _valid(g):
                prio.append((0.9, g))
                tags.setdefault(_key(g), "dsl")
        # 3. augmented solutions
        for k, prog in aug_progs:
            g = apply_program(prog, dihedral(k)(tg))
            if g is not None:
                g = dihedral(DIHEDRAL_INV[k])(g)
            if _valid(g):
                prio.append((0.8, g))
                tags.setdefault(_key(g), "aug")
        # 4. partial programs
        for pp in partial_progs[:2]:
            g = apply_program(pp, tg)
            if _valid(g):
                prio.append((0.5, g))
                tags.setdefault(_key(g), "partial")
        # 5. priors
        for g in infer_fallbacks(demos, tg):
            prio.append((0.3, g))
            tags.setdefault(_key(g), "prior")

        prio = _distinct([g for _, g in sorted(prio, key=lambda x: -x[0])])
        per_test[i] = prio[:2]
    if return_tags:
        tags_out = [[tags.get(_key(g), "other") for g in per_test[i]] for i in range(len(tests))]
        return per_test, tags_out
    return per_test


def _key(g):
    return (g.shape, g.tobytes())


# ---------------- run loop ----------------
def load_challenges_kaggle(input_dir):
    """Load challenges from a Kaggle competition data dir (or local JSON bundle)."""
    path = os.path.join(input_dir, "arc-agi_test-challenges.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        raw = json.load(f)
    return raw


def fill_submission(sample_sub, challenges, predictor, cfg, log=print):
    """Iterate tasks, call predictor, produce submission dict."""
    sub = {}
    ids = sorted(challenges.keys())
    total = len(ids)
    budget = cfg["total_budget"] * (1 - cfg["safety_margin"])
    t_start = time.monotonic()
    solved_stats = {"heur": 0, "dsl": 0, "aug": 0, "prior": 0}

    for n, tid in enumerate(ids):
        remaining = budget - (time.monotonic() - t_start)
        tasks_left = total - n
        per_task = max(3.0, remaining / max(tasks_left, 1))
        deadline_task = time.monotonic() + min(per_task, cfg["dsl_deadline"] + cfg["aug_deadline"] + cfg["heur_deadline"] + 5)

        task = challenges[tid]
        tests = task.get("test", [])
        preds = predictor(task, deadline_task)

        entries = []
        for j in range(len(tests)):
            outs = preds[j] if j < len(preds) else []
            if not outs:
                tg = tests[j].get("input") if isinstance(tests[j], dict) else tests[j]
                g = to_grid(tg)
                outs = [g] if g is not None else [np.zeros((1, 1), dtype=np.int8)]
            while len(outs) < 2:
                outs.append(outs[0])
            outs = outs[:2]
            attempts = {}
            for a in range(2):
                attempts[f"attempt_{a + 1}"] = from_grid(outs[a])
            entries.append(attempts)
        sub[tid] = entries

        if (n + 1) % cfg["progress_every"] == 0 or (n + 1) == total:
            el = time.monotonic() - t_start
            log(f"[{n + 1}/{total}] elapsed={el / 60:.1f}m remaining~{max(0, budget - el) / 60:.1f}m")
            try:
                with open(sub_path_global, "w") as f:
                    json.dump({**sample_sub, **sub} if sample_sub else sub, f)
            except Exception:
                pass
    return sub


def validate_submission(sub):
    """Sanity-check structure; coerce to serializable form."""
    problems = []
    for tid, entries in sub.items():
        if not isinstance(entries, list) or len(entries) == 0:
            problems.append(f"{tid}: no entries")
            continue
        for e in entries:
            if not isinstance(e, dict) or "attempt_1" not in e or "attempt_2" not in e:
                problems.append(f"{tid}: missing attempts")
                continue
            for k in ("attempt_1", "attempt_2"):
                g = e[k]
                if not isinstance(g, list) or len(g) == 0 or not (1 <= len(g) <= MAX_SIDE):
                    problems.append(f"{tid}/{k}: bad shape")
                    continue
                w = len(g[0])
                if not (1 <= w <= MAX_SIDE):
                    problems.append(f"{tid}/{k}: bad width")
                for row in g:
                    if len(row) != w:
                        problems.append(f"{tid}/{k}: ragged")
                        break
                    for v in row:
                        if not isinstance(v, int) or v < 0 or v > 9:
                            problems.append(f"{tid}/{k}: bad value {v}")
                            break
    return problems


sub_path_global = "/kaggle/working/submission.json"


def run(input_dir, out_path, cfg=None, sample_sub=None, log=print, challenges=None):
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    global sub_path_global
    sub_path_global = out_path

    if challenges is None:
        challenges = load_challenges_kaggle(input_dir)
    if challenges is None:
        raise RuntimeError(f"arc-agi_test-challenges.json not found in {input_dir}")

    def predictor(task, deadline_task):
        return solve_task(task, cfg, deadline_task)

    log(f"Solving {len(challenges)} tasks ...")
    t0 = time.monotonic()
    sub = fill_submission(sample_sub, challenges, predictor, cfg, log=log)
    log(f"Done in {(time.monotonic() - t0) / 60:.1f} min")

    problems = validate_submission(sub)
    if problems:
        log(f"WARNING: {len(problems)} validation issues; first few: {problems[:5]}")

    final = {**sample_sub, **sub} if sample_sub else sub
    with open(out_path, "w") as f:
        json.dump(final, f)
    log(f"Submission written to {out_path} ({len(sub)} tasks)")
    return sub


### 6. Main run — produce `submission.json`

Loads the hidden test challenges, runs the solver under the time budget, and writes
`submission.json` in the exact required format (2 attempts per test input).
A checkpoint is written every 10 tasks so partial progress is never lost.


In [ ]:
# --------------------------- load data & run --------------------------------
def load_challenges():
    candidates = ("arc-agi_test-challenges.json", "arc-agi_test_challenges.json",
                  "test_challenges.json")
    path = next((os.path.join(INPUT_DIR, n) for n in candidates
                 if os.path.exists(os.path.join(INPUT_DIR, n))), None)
    if path is None:
        present = sorted(os.listdir(INPUT_DIR))[:20] if os.path.isdir(INPUT_DIR) else []
        raise RuntimeError(
            "arc-agi_test-challenges.json not found in " + str(INPUT_DIR) +
            "; files present: " + (", ".join(present) or "<none>"))
    with open(path) as f:
        challenges = json.load(f)
    if QUICK_TEST_N:
        ids = sorted(challenges.keys())[:QUICK_TEST_N]
        challenges = {k: challenges[k] for k in ids}
    return challenges

def load_sample_submission():
    for name in ("sample_submission.json",):
        p = os.path.join(INPUT_DIR, name)
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)
    return None

OUT_PATH = "/kaggle/working/submission.json" if os.path.isdir("/kaggle/working") else "submission.json"

challenges = load_challenges()
sample_sub = load_sample_submission()
print(f"Tasks to solve: {len(challenges)} | sample submission: {'yes' if sample_sub else 'no'}")

sub = run(INPUT_DIR, OUT_PATH, cfg=CFG, sample_sub=sample_sub, challenges=challenges, log=print)

# final integrity check
problems = validate_submission(sub)
final_obj = {**sample_sub, **sub} if sample_sub else sub
with open(OUT_PATH, "w") as f:
    json.dump(final_obj, f)
print("Validation issues:", len(problems), problems[:5])
print(f"Wrote {OUT_PATH} with {len(final_obj)} tasks.")


## Notes & extension ideas

- **Where the score comes from:** verified heuristics and short DSL programs solve the
  "mechanical" subset of ARC-AGI-2; augmented search adds symmetry-robustness. Everything is
  verified against the demos, so precision is high — false positives are structurally impossible.
- **Tuning knobs:** `DSL_SECONDS` / `AUG_SECONDS` / `BEAM_WIDTH` trade runtime vs. coverage.
  Within the 9 h limit, raising `MAX_DEPTH` to 7 and `BEAM_WIDTH` to ~300 usually helps slightly.
- **Ethics & rules:** no internet, no external models, no test-set leakage, no answer hardcoding —
  predictions derive only from each task's demonstrations. Public sharing of this notebook is
  permitted and encouraged by the competition rules (post it on the competition forum).
- **Next-level ideas:** (1) learn an op-prior from the 1000 training tasks and reorder the beam;
  (2) add two-grid relational ops (overlay, XOR, alignment); (3) attach an open-source
  ARC-fine-tuned model as a Kaggle dataset and ensemble its verified outputs with this engine.
